In [4]:
import os
import xml.etree.ElementTree as ET
import random
import shutil

# ================= CONFIG =================
SOURCE_DIR = r"C:\Users\100ra\Downloads\Combine_One"
OUTPUT_DIR = r"C:\Users\100ra\Downloads\dataset"
TRAIN_RATIO = 0.8   # 80% train, 20% val
CLASS_NAME = "license_plate"
# ==========================================

# Create YOLO folder structure
for split in ["train", "val"]:
    os.makedirs(os.path.join(OUTPUT_DIR, "images", split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "labels", split), exist_ok=True)

# Collect images
images = [f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(".jpg")]
random.shuffle(images)

split_index = int(len(images) * TRAIN_RATIO)
train_images = images[:split_index]
val_images = images[split_index:]

def convert_bbox(img_w, img_h, xmin, ymin, xmax, ymax):
    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h
    return x_center, y_center, width, height

def process_split(image_list, split_name):
    for img_name in image_list:
        img_path = os.path.join(SOURCE_DIR, img_name)
        xml_path = os.path.join(SOURCE_DIR, img_name.replace(".jpg", ".xml"))

        if not os.path.exists(xml_path):
            print(f"[SKIPPED] XML missing for {img_name}")
            continue

        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
        except:
            print(f"[SKIPPED] Corrupted XML: {xml_path}")
            continue

        size = root.find("size")
        img_w = int(size.find("width").text)
        img_h = int(size.find("height").text)

        yolo_labels = []

        for obj in root.findall("object"):
            # IMPORTANT: Ignore <name> text (it is plate number)
            cls_id = 0  # license_plate

            bndbox = obj.find("bndbox")
            xmin = float(bndbox.find("xmin").text)
            ymin = float(bndbox.find("ymin").text)
            xmax = float(bndbox.find("xmax").text)
            ymax = float(bndbox.find("ymax").text)

            x, y, w, h = convert_bbox(img_w, img_h, xmin, ymin, xmax, ymax)
            yolo_labels.append(f"{cls_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

        if not yolo_labels:
            print(f"[SKIPPED] No objects in {img_name}")
            continue

        # Save label
        label_path = os.path.join(
            OUTPUT_DIR, "labels", split_name, img_name.replace(".jpg", ".txt")
        )
        with open(label_path, "w") as f:
            f.write("\n".join(yolo_labels))

        # Copy image
        shutil.copy(
            img_path,
            os.path.join(OUTPUT_DIR, "images", split_name, img_name)
        )

# Process dataset
process_split(train_images, "train")
process_split(val_images, "val")

# Create data.yaml
yolo_path = OUTPUT_DIR.replace("\\", "/")
yaml_content = f"""
path: {yolo_path}
train: images/train
val: images/val

names:
  0: {CLASS_NAME}
"""

with open(os.path.join(OUTPUT_DIR, "data.yaml"), "w") as f:
    f.write(yaml_content.strip())

print("✅ YOLOv8 dataset prepared successfully!")


[SKIPPED] Corrupted XML: C:\Users\100ra\Downloads\Combine_One\018b52e6-e9a1-42c2-8ce7-0617e8c8e021___3e7fd381-0ae5-4421-8a70-279ee0ec1c61_sbtb02_auto1.JPG
✅ YOLOv8 dataset prepared successfully!
